# Session 5 — MDP inventory (define the framework, then run the pipeline)

**Salih Tutun, PhD** · DAT 5668 · Module 3 · Fundamentals → define pieces → lab pipeline

Hey. You do **not** need to know RL already. Tonight we do two things only:

1. **DEFINE** the named pieces of the agent framework (same loop as Session 1).  
2. **USE them one by one** in a pipeline (and play for 10–15 minutes).

Course map (keep this forever): `Students/Agent-Framework.md`

---

## The story (zero jargon)

A **Home and lifestyle** shelf:

- morning stock = **state** $s$
- how many you order = **action** $a$
- customers = **demand** $d$ (real invoices)
- money / penalty = **reward** $r$
- leftover for tomorrow = **next state** $s'$

Session 4's bandit **forgot** the shelf. Tonight the shelf **remembers**. That memory is why we call it an **MDP**.

![Shelf as an MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

---

---

## Big picture (read this diagram first)

![Agent loop for Session 5](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/session05-loop.png)

*Diagrams load from GitHub (same pattern as Sessions 1–4). If a picture is blank, hard-refresh or File → Open notebook again.*

Same loop as Session 1. Tonight we **DEFINE** those boxes, then run them in a **PIPELINE**.

## Framework tonight (Session 1 words → code names)

| Step you say | RL name | Function we DEFINE |
| --- | --- | --- |
| Load data | environment data | `load_dataset` |
| Observe | state $s$ | `observe` |
| Legal moves | action set | `list_actions` |
| Decide | policy | `decide` / `policy` |
| Environment | transition | `next_state` (also called `transition`) |
| Feedback | reward $r$ | `reward` |
| One day | step | `run_day` |
| Many days | episode | `run_episode` |

```text
load_dataset
  → observe(s)
  → decide(s) → a
  → next_state(s, a, d) → s'
  → reward(s, a, d) → r
  → s ← s'   (repeat)
```

**Student rule:** define the functions first. Then wire the pipeline. Do not hide the loop in one mystery cell.

---

## How class feels tonight

| Block | What happens |
| --- | --- |
| A | Fundamentals + Stop 1 (words) |
| B | **By hand first** — formulation + Days A/B/C |
| C | **DEFINE** + **PIPELINE** (code matches hand work) |
| D | **LAB 10–15 min** — you try; I walk around |
| E | Debrief + framework sentence |

---

## Run of show
1. Slides (notebook closed).
2. Stop 1 — name what the bandit missed.
3. **By hand first** — formulation + Days A/B/C on paper.
4. DEFINE + PIPELINE with me (code matches your hand work).
5. LAB challenges (I walk the room).
6. Come back — stockout check + framework line.


## How we work

| Cue | What you do |
| --- | --- |
| **DEFINE** | We write/run a named function. Read the comment header. |
| **PIPELINE** | We call those functions **in order**. |
| **TRY THIS** | Change `<<< CHANGE ME`. Re-run. Talk. |
| **LAB** | 10–15 minutes. I walk around. |
| **RAISE HAND** | Checklist done — or stuck >2 minutes. |

Partner rule: one laptop; both can explain the last printout using framework words ($s$, $a$, $r$, $s'$).


# Part A — Fundamentals check

> Slides first. Then one sentence. Then we **define** code.


## Stop 1 — OPEN · PAIR

1. What could Session 4's bandit **not** remember?  
2. What number is the **state** on this shelf tonight?

Type short phrases.


In [ ]:
# ============================================================
# STOP 1 — FILL (words first, before formulas)
# Goal: name the framework idea in plain English.
# ============================================================

# What did Session 4's bandit forget? (stock / shelf memory)
bandit_missed = ""  # <<< YOU TYPE HERE

# What is state s tonight? (units on the shelf this morning)
state_tonight = ""  # <<< YOU TYPE HERE

# Print your answers (or a hint if still empty)
print("Bandit missed:", bandit_missed or "(type it)")
print("State tonight:", state_tonight or "(type it)")


# Part B — Formulation, then calculate Day 1 → Day 2 → Day 3 by hand

> Notebook open. **Do not scroll into the DEFINE code yet.**
> First we write the **formulation** (symbols + equations).
> Then **you calculate** Day 1, Day 2, and Day 3 **one number at a time**.

## Why by hand?

If you only read Python, the MDP feels like "a bunch of functions."
If you compute **three days by hand**, you already understand tonight:

observe $s$ → decide $a$ → demand $d$ → next state $s'$ → reward $r$

---

## Big picture — one day

![One day by hand](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/session05-one-day.png)

Keep this diagram open while you calculate.

---

## 0) Notation card (say each symbol once)

| Symbol | Say it | Meaning tonight | Values |
| --- | --- | --- | --- |
| $s$ | state | Morning stock | $\{0,1,2,3,4\}$ |
| $a$ | action | Morning order | $\{0,1,2\}$ |
| $d$ | demand | Customers today | $\{0,1,2\}$ (binned) |
| $s'$ | next state | Tomorrow morning stock | $\{0,1,2,3,4\}$ |
| $r$ | reward | Today's score | real number |
| $C$ | capacity | Shelf max | $C=4$ |
| $P(d)$ | pee of dee | Chance of each demand | from data later |
| $\gamma$ | gamma | Discount on future | $0.95$ |
| $\pi(s)$ | policy / decide | Rule that picks $a$ from $s$ | base-stock toward 2 |

**Business knobs**

| Knob | Value | Role |
| --- | --- | --- |
| Capacity $C$ | 4 | $\mathrm{started}=\min(C,s+a)$ |
| Margin | 2.0 | $+\,2.0\cdot\mathrm{sales}$ in $r$ |
| Hold cost | 0.2 | $-\,0.2\cdot s'$ in $r$ |
| Stockout penalty | 1.5 | $-\,1.5\cdot\mathrm{stockout}$ in $r$ |

---

## 1) Formulation (equations — same every day)

**MDP pieces:** state $s$, action $a$, transition via demand $d$, reward $r$, discount $\gamma$.

**Shelf physics**

$$
\begin{aligned}
\mathrm{started} &= \min(C,\, s+a) \\
\mathrm{sales} &= \min(\mathrm{started},\, d) \\
s' &= \mathrm{started}-\mathrm{sales} \\
\mathrm{stockout} &= \max(0,\, d-\mathrm{started})
\end{aligned}
$$

**Reward**

$$
r = 2.0\cdot\mathrm{sales} - 0.2\cdot s' - 1.5\cdot\mathrm{stockout}
$$

**Policy (when we decide)**

$$
a = \pi(s) = \max\!\big(0,\, \min(2,\, 2-s)\big)
$$

**Pipeline order**

```text
observe s → decide a → environment gives d → next_state → s' → reward → r → s ← s'
```

Now calculate **Day 1**, then **Day 2**, then **Day 3**. Fill **one line at a time**.


## Day 1 — empty shelf must hurt

**Transition physics (same every day)**

1. **started** $=\min(C,\, s+a)$ — units available after the morning order (cannot exceed capacity)
2. **sales** $=\min(\mathrm{started},\, d)$ — cannot sell more than we have or customers want
3. **next state** $s'=\mathrm{started}-\mathrm{sales}$ — leftover becomes tomorrow's morning stock
4. **stockout** $=\max(0,\, d-\mathrm{started})$ — customers we failed to serve

**Reward**

$$
r = 2.0\cdot\mathrm{sales} - 0.2\cdot s' - 1.5\cdot\mathrm{stockout}
$$

**Given numbers for Day 1:** $s=0$, $a=0$, $d=2$, $C=4$

| Number | Symbol | Meaning |
| --- | --- | --- |
| 0 | $s$ | morning stock (empty) |
| 0 | $a$ | order nothing |
| 2 | $d$ | two customers |
| 4 | $C$ | capacity |

**Plug into the equations one line at a time** (paper or FILL cell):

1. $\mathrm{started}=\min(C,s+a)=\min(4,\,0+0)=\;?$  
2. $\mathrm{sales}=\min(\mathrm{started},d)=\;?$  
3. $s'=\mathrm{started}-\mathrm{sales}=\;?$  
4. $\mathrm{stockout}=\max(0,\,d-\mathrm{started})=\;?$  
5. $r=2.0\cdot\mathrm{sales}-0.2\cdot s'-1.5\cdot\mathrm{stockout}=\;?$

**Story to expect:** no stock, no order, two customers → stockout pain, no margin.


In [ ]:
# ============================================================
# DAY 1 — FILL (calculate one line at a time)
# Given: s=0, a=0, d=2, C=4
# Equations:
#   started = min(C, s + a)
#   sales   = min(started, d)
#   s'      = started - sales
#   stockout= max(0, d - started)
#   r       = 2.0*sales - 0.2*s' - 1.5*stockout
# ============================================================

C, s, a, d = 4, 0, 0, 2

started = None   # <<< FILL  min(C, s + a)
sales = None     # <<< FILL  min(started, d)
s_next = None    # <<< FILL  started - sales
stockout = None  # <<< FILL  max(0, d - started)
r = None         # <<< FILL  2.0*sales - 0.2*s_next - 1.5*stockout

print("Day 1 your numbers:")
print(f"  started={started}  sales={sales}  s'={s_next}  stockout={stockout}  r={r}")

# Optional self-check (uncomment after you fill):
# assert (started, sales, s_next, stockout, r) == (0, 0, 0, 2, -3.0)
# print("Day 1 OK")


## Day 2 — we stocked the shelf

**Transition physics (same every day)**

1. **started** $=\min(C,\, s+a)$ — units available after the morning order (cannot exceed capacity)
2. **sales** $=\min(\mathrm{started},\, d)$ — cannot sell more than we have or customers want
3. **next state** $s'=\mathrm{started}-\mathrm{sales}$ — leftover becomes tomorrow's morning stock
4. **stockout** $=\max(0,\, d-\mathrm{started})$ — customers we failed to serve

**Reward**

$$
r = 2.0\cdot\mathrm{sales} - 0.2\cdot s' - 1.5\cdot\mathrm{stockout}
$$

**Given numbers for Day 2:** $s=1$, $a=2$, $d=2$, $C=4$

| Number | Symbol | Meaning |
| --- | --- | --- |
| 1 | $s$ | start with 1 on shelf |
| 2 | $a$ | order 2 |
| 2 | $d$ | two customers |
| 4 | $C$ | capacity |

**Plug into the same equations:**

1. $\mathrm{started}=\min(4,\,1+2)=\;?$  
2. $\mathrm{sales}=\min(\mathrm{started},2)=\;?$  
3. $s'=\mathrm{started}-\mathrm{sales}=\;?$  
4. $\mathrm{stockout}=\max(0,\,2-\mathrm{started})=\;?$  
5. $r=2.0\cdot\mathrm{sales}-0.2\cdot s'-1.5\cdot\mathrm{stockout}=\;?$

**Story to expect:** order into stock, sell 2, maybe hold leftover overnight.


In [ ]:
# ============================================================
# DAY 2 — FILL (calculate one line at a time)
# Given: s=1, a=2, d=2, C=4
# Same equations as Day 1
# ============================================================

C, s, a, d = 4, 1, 2, 2

started = None   # <<< FILL  min(C, s + a)
sales = None     # <<< FILL  min(started, d)
s_next = None    # <<< FILL  started - sales
stockout = None  # <<< FILL  max(0, d - started)
r = None         # <<< FILL  2.0*sales - 0.2*s_next - 1.5*stockout

print("Day 2 your numbers:")
print(f"  started={started}  sales={sales}  s'={s_next}  stockout={stockout}  r={r}")

# Optional self-check (uncomment after you fill):
# assert (started, sales, s_next, stockout, abs(r - 3.8) < 1e-9)
# print("Day 2 OK")


## Day 3 — decide first, then transition

**Transition physics (same every day)**

1. **started** $=\min(C,\, s+a)$ — units available after the morning order (cannot exceed capacity)
2. **sales** $=\min(\mathrm{started},\, d)$ — cannot sell more than we have or customers want
3. **next state** $s'=\mathrm{started}-\mathrm{sales}$ — leftover becomes tomorrow's morning stock
4. **stockout** $=\max(0,\, d-\mathrm{started})$ — customers we failed to serve

**Reward**

$$
r = 2.0\cdot\mathrm{sales} - 0.2\cdot s' - 1.5\cdot\mathrm{stockout}
$$

**Given for Day 3:** morning $s=1$, policy target $=2$, then suppose $d=2$, $C=4$.

**First decide** (then use $a$ in the physics above):

$$
a=\pi(s)=\max\!\big(0,\,\min(2,\,2-s)\big)=\max\!\big(0,\,\min(2,\,2-1)\big)=\;?
$$

**Then plug into the transition physics:**

1. $\mathrm{started}=\min(4,\,1+a)=\;?$  
2. $\mathrm{sales}=\min(\mathrm{started},2)=\;?$  
3. $s'=\;?$  
4. $\mathrm{stockout}=\;?$  
5. $r=\;?$

Partner: cover answers; both must agree before running the check.


In [ ]:
# ============================================================
# DAY 3 — FILL (decide first, then physics)
# Given: s=1, target=2, d=2, C=4
# decide: a = max(0, min(2, target - s))
# then same started / sales / s' / stockout / r equations
# ============================================================

C, s, target, d = 4, 1, 2, 2

a = None         # <<< FILL  max(0, min(2, target - s))
started = None   # <<< FILL  min(C, s + a)
sales = None     # <<< FILL  min(started, d)
s_next = None    # <<< FILL  started - sales
stockout = None  # <<< FILL  max(0, d - started)
r = None         # <<< FILL  2.0*sales - 0.2*s_next - 1.5*stockout

print("Day 3 your numbers:")
print(f"  a={a}  started={started}  sales={sales}  s'={s_next}  stockout={stockout}  r={r}")

# Optional self-check (uncomment after you fill):
# assert (a, started, sales, s_next, stockout, r) == (1, 2, 2, 0, 0, 4.0)
# print("Day 3 OK")


## Check Days 1–3 together (after your FILLs)

Run the next cell only after your group has numbers for Day 1, Day 2, and Day 3.
It reprints the formulation with **labeled plug-ins** so you can compare.


In [ ]:
# ============================================================
# CHECK Days 1–3 — RUN WITH ME (after student FILLs)
# Shows equation → plug-in with every number labeled.
# ============================================================

CAPACITY = 4
MARGIN = 2.0
HOLD_COST = 0.2
STOCKOUT_PENALTY = 1.5


def walk_one_day(day_num, s, a, d, explain_decide=False, target=2):
    print("=" * 60)
    print(f"Day {day_num}")
    print("=" * 60)

    if explain_decide:
        print("Step 1  observe")
        print("         equation: s = morning stock")
        print(f"         plug-in:  s = {s}")
        print()
        a_calc = max(0, min(2, target - s))
        print("Step 2  decide")
        print("         equation: a = max(0, min(2, target - s))")
        print(f"         plug-in:  a = max(0, min(2, {target} - {s})) = {a_calc}")
        assert a == a_calc
        print()
        print("Step 3  demand from environment")
        print("         equation: d = customers today")
        print(f"         plug-in:  d = {d}")
        print()
        step0 = 4
    else:
        print(f"Given: s={s} (stock), a={a} (order), d={d} (demand), C={CAPACITY} (capacity)")
        print()
        step0 = 1

    started = min(CAPACITY, s + a)
    print(f"Step {step0}  started = min(C, s + a)")
    print(f"         plug-in: min({CAPACITY}, {s} + {a}) = {started}")
    print(f"         labels: C={CAPACITY}, s={s}, a={a}")
    print()

    sales = min(started, d)
    print(f"Step {step0 + 1}  sales = min(started, d)")
    print(f"         plug-in: min({started}, {d}) = {sales}")
    print()

    s_next = started - sales
    print(f"Step {step0 + 2}  s' = started - sales")
    print(f"         plug-in: {started} - {sales} = {s_next}")
    print()

    stockout = max(0, d - started)
    print(f"Step {step0 + 3}  stockout = max(0, d - started)")
    print(f"         plug-in: max(0, {d} - {started}) = {stockout}")
    print()

    r = MARGIN * sales - HOLD_COST * s_next - STOCKOUT_PENALTY * stockout
    print(f"Step {step0 + 4}  r = 2.0*sales - 0.2*s' - 1.5*stockout")
    print(f"         plug-in: 2.0*{sales} - 0.2*{s_next} - 1.5*{stockout} = {r:.1f}")
    print()
    print(f">>> Day {day_num} result: s'={s_next}, r={r:.1f}")
    print()
    return r


assert abs(walk_one_day(1, 0, 0, 2) - (-3.0)) < 1e-9
assert abs(walk_one_day(2, 1, 2, 2) - 3.8) < 1e-9
assert abs(walk_one_day(3, 1, 1, 2, explain_decide=True) - 4.0) < 1e-9
print("Targets: Day 1 → r=-3.0 | Day 2 → r=3.8 | Day 3 → r=4.0")
print("If your FILL matches, you are ready for DEFINE (same math, named functions).")


## What code will do next (preview only)

| After Days 1–3, code will… | Framework name |
| --- | --- |
| Load real $P(d)$ from invoices | `load_dataset` |
| Wrap the formulas you just used | `observe`, `decide`, `next_state`, `reward` |
| Call them in order for many days | `run_day`, `run_episode` |

When Day 1 / Day 2 / Day 3 feel mechanical, scroll to **DEFINE** — you are not learning new math, only naming it.


# Part C — DEFINE the framework in code (RUN WITH ME)

> You already calculated Day 1–3 by hand. Now we **name** the same formulas as functions.

We define **one function at a time**. Later sessions add pieces (Q-update, deep net) — they do **not** rename this loop.

## Equations ↔ function names (say before you run)

| Function | Equation / rule from Part B |
| --- | --- |
| `load_dataset()` | Build $P(d)$ for $d\in\{0,1,2\}$ from Home-and-lifestyle invoices |
| `observe(stock)` | $s\leftarrow$ morning stock |
| `list_actions()` | $\mathcal{A}=\{0,1,2\}$ |
| `decide(s)` / `policy(s)` | $a=\max(0,\min(2,2-s))$ |
| `next_state(s,a,d)` | $\mathrm{started}=\min(C,s+a)$; $\mathrm{sales}=\min(\mathrm{started},d)$; $s'=\mathrm{started}-\mathrm{sales}$ |
| `reward(s,a,d)` | $r=2.0\cdot\mathrm{sales}-0.2\cdot s'-1.5\cdot\max(0,d-\mathrm{started})$ |
| `expected_reward(s,a)` | $R(s,a)=\sum_d P(d)\,r(s,a,d)$ |

**Business knobs in code**

| Knob | Value | Role |
| --- | --- | --- |
| `CAPACITY` | 4 | $C$ |
| `MARGIN` | 2.0 | profit per sale |
| `HOLD_COST` | 0.2 | cost per leftover |
| `STOCKOUT_PENALTY` | 1.5 | pain per miss |
| `GAMMA` | 0.95 | $\gamma$ (named tonight; used heavily later) |


## Setup — find the dataset (Colab Drive or local)

Same habit as Session 4: shared folder `MyDrive/DAT5668/data/` with `supermarket-sales.csv`.


In [ ]:
# ============================================================
# SETUP — RUN WITH ME
# Goal: find supermarket-sales.csv (Google Drive in Colab, or local files).
# ============================================================

# Path tools: build file paths in a clean way
from pathlib import Path

# Where the shared course data folder should live in Colab
DRIVE_ROOT = Path("/content/drive/MyDrive/DAT5668/data")

# Try to import Colab's Drive helper (fails on a normal laptop — that is OK)
try:
    from google.colab import drive  # type: ignore

    # Attach your Google Drive to this Colab runtime (asks permission once)
    drive.mount("/content/drive", force_remount=False)
    # Confirm mount worked
    print("Drive mounted.")
except ImportError:
    # Not running in Colab — we will look for local course CSVs instead
    print("Not Colab — OK. We will use local course data if present.")

# If the Drive folder exists, list CSVs so we can see supermarket-sales.csv
if DRIVE_ROOT.exists():
    # Collect every .csv file name in the shared data folder
    files = sorted(p.name for p in DRIVE_ROOT.glob("*.csv"))
    # Show what we found
    print("CSVs:", files)
    # Explicit yes/no for the file Session 5 needs
    print("supermarket-sales.csv →", "YES" if "supermarket-sales.csv" in files else "MISSING")
else:
    # Folder missing: fine locally; in Colab means mount/share is not ready
    print("Drive folder not found yet (OK if local):", DRIVE_ROOT)


In [ ]:
# ============================================================
# DEFINE — RUN WITH ME
# Rule: define named framework pieces FIRST.
# Later the PIPELINE cell will call them ONE BY ONE.
# Names match Students/Agent-Framework.md
# Read each comment. Every line earns its place.
# ============================================================

# --- imports (tools we need) ---
from pathlib import Path                 # build file paths
from collections import Counter, defaultdict  # count / group numbers easily
import csv                               # read the supermarket CSV rows
import numpy as np                       # argmax / arrays for charts later
import matplotlib.pyplot as plt          # bar charts and stock-path plots

# --- business knobs (say these out loud once) ---
CAPACITY = 4            # shelf cannot hold more than 4 units
HOLD_COST = 0.2         # cost for each unit left overnight
MARGIN = 2.0            # profit for each unit we sell
STOCKOUT_PENALTY = 1.5  # pain for each customer we cannot serve
GAMMA = 0.95            # tomorrow's cash counts a bit less than today's (board word)

# Colab path to shared data (ignored automatically if that folder is missing)
DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"


def course_data(name: str) -> Path:
    # Helper: find a CSV on Drive first, then in local course folders.
    candidates = []  # list of places we will look
    if DRIVE_DATA_FOLDER:
        # Prefer the shared Drive folder when present
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    # Also try common local locations (laptop / GitHub clone / Canvas zip)
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data"), Path("../data")):
        candidates.append(base / name)
    # Return the first path that actually exists
    for path in candidates:
        if path.exists():
            print("Using:", path)  # so everyone sees which file loaded
            return path
    # If nothing worked, stop with a clear error (path issue, not MDP issue)
    raise FileNotFoundError("Could not find " + name)


# ----- 1) DEFINE load_dataset -----
def load_dataset():
    # DEFINE: load_dataset
    # Turn real Home-and-lifestyle invoices into chances P(demand) for d = 0,1,2.
    daily = defaultdict(int)  # date → total quantity that day
    # Open the CSV with UTF-8 text encoding
    with course_data("supermarket-sales.csv").open(encoding="utf-8") as handle:
        # DictReader gives each row as {column_name: value}
        for row in csv.DictReader(handle):
            # Keep only the Home and lifestyle product line
            if (row.get("Product line") or "").strip() != "Home and lifestyle":
                continue  # skip other product lines
            # Add this row's quantity into that calendar day's total
            daily[row["Date"]] += int(float(row["Quantity"]))
    # Sorted list of daily totals (one number per day that had sales)
    totals = sorted(daily.values())
    if not totals:
        # Safety fallback if the file were empty (should not happen in class)
        return {0: 1 / 3, 1: 1 / 3, 2: 1 / 3}
    # Split days into low / mid / high thirds → labels 0, 1, 2 (keeps MDP readable)
    q1, q2 = totals[len(totals) // 3], totals[(2 * len(totals)) // 3]
    bins = [0 if qty <= q1 else 1 if qty <= q2 else 2 for qty in totals]
    counts = Counter(bins)  # how many days fell in each bin
    n = len(bins)           # total days
    # Convert counts → probabilities that sum to 1
    return {level: counts[level] / n for level in (0, 1, 2)}


# Run load_dataset once and keep the result as the environment demand model
DEMAND_P = load_dataset()


# ----- 2) DEFINE observe -----
def observe(stock):
    # DEFINE: observe
    # Session 1 word OBSERVE → state s
    # Tonight the state IS on-hand stock; we still wrap it so the pipeline is explicit.
    return int(stock)  # force integer units


# ----- 3) DEFINE list_actions -----
def list_actions():
    # DEFINE: list_actions
    # Legal morning orders the warehouse can perform tonight.
    return [0, 1, 2]


# ----- 4) DEFINE decide / policy -----
def decide(state, target=2):
    # DEFINE: decide (also called policy)
    # Session 1 word DECIDE → map state s to action a
    # Rule tonight: try to bring shelf up to `target`, but never order more than 2.
    stock = observe(state)                 # read s
    return max(0, min(2, target - stock))  # order needed, clipped to [0, 2]


def policy(state, target=2):
    # Alias for decide — same framework slot (two names, one job)
    return decide(state, target=target)


# ----- 5) DEFINE next_state / transition -----
def next_state(state, action, demand):
    # DEFINE: next_state (also called transition)
    # Session 1 feedback includes next state s'
    # Physics for one day on the shelf:
    started = min(CAPACITY, observe(state) + int(action))  # units available after order
    sales = min(started, int(demand))                      # cannot sell more than we have or customers want
    s_next = started - sales                               # leftover = tomorrow's state
    return s_next, sales


def transition(state, action, demand):
    # Alias for next_state — same framework slot
    return next_state(state, action, demand)


# ----- 6) DEFINE reward -----
def reward(state, action, demand):
    # DEFINE: reward
    # Session 1 feedback includes reward r
    s = observe(state)  # starting stock
    a = int(action)     # order
    d = int(demand)     # customers
    s_next, sales = next_state(s, a, d)                 # need sales and leftover
    stockout = max(0, d - min(CAPACITY, s + a))         # unmet customers
    # money from sales, minus holding leftover, minus stockout pain
    return MARGIN * sales - HOLD_COST * s_next - STOCKOUT_PENALTY * stockout


# ----- 7) DEFINE expected_reward (helper to compare actions — not a new loop) -----
def expected_reward(state, action):
    # Average reward over real demand chances P(d)
    # Formula: sum_d  P(d) * reward(s, a, d)
    return sum(p * reward(state, action, d) for d, p in DEMAND_P.items())


# --- sanity prints so the room sees the DEFINE step worked ---
print("DEFINED: load_dataset, observe, list_actions, decide/policy, next_state/transition, reward")
print("DEMAND_P:", {k: round(v, 3) for k, v in DEMAND_P.items()})
print("Legal actions:", list_actions())
print("Example decide(stock=1) →", decide(1))


## Big picture — DEFINE then PIPELINE

![DEFINE then PIPELINE](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/session05-define-pipeline.png)

You do **not** invent a new loop in the lab — you call these same boxes.


## PIPELINE — use the pieces one by one

> Same order as Day 1–3: observe → decide → next_state → reward.

We **do not invent new logic**. We only call what we defined.

### `run_day(s, a, d)` — one morning

$$
s\leftarrow\mathrm{observe}(s),\quad
(s',\mathrm{sales})\leftarrow\mathrm{next\_state}(s,a,d),\quad
r\leftarrow\mathrm{reward}(s,a,d)
$$

### `run_episode` — many mornings (MDP memory)

$$
a_t=\mathrm{decide}(s_t)
\;\rightarrow\;
\mathrm{run\_day}
\;\rightarrow\;
s_{t+1}\text{ becomes next morning's }s
$$

**Teach aloud:** point at column $s'$ and the next row's $s$ — that arrow **is** the MDP.


In [ ]:
# ============================================================
# PIPELINE — RUN WITH ME
# Rule: do NOT invent new logic here.
# Only call DEFINE pieces in Session-1 order:
#   observe → decide → next_state → reward → (repeat)
# ============================================================

def run_day(state, action, demand):
    # One environment step (one business day).
    # 1) OBSERVE state s
    s = observe(state)
    # 2) action a was already chosen by decide (passed in)
    a = int(action)
    # 3) demand d comes from the environment (passed in)
    d = int(demand)
    # 4) TRANSITION: get next state s' and sales
    s_next, sales = next_state(s, a, d)
    # 5) REWARD: score this day
    r = reward(s, a, d)
    # 6) Return a small dictionary so prints are easy to read
    return {
        "s": s,            # state we observed
        "a": a,            # action we took
        "d": d,            # demand that happened
        "sales": sales,    # units sold
        "r": r,            # reward
        "s_next": s_next,  # next state (tomorrow's stock)
    }


def run_episode(start_stock=1, target=2, demands=None):
    # Many days: repeat the pipeline and CARRY stock memory forward.
    if demands is None:
        # Fixed demand path so the whole class sees the same story
        demands = [1, 2, 0, 2, 1]
    # Day-1 morning: observe starting stock
    s = observe(start_stock)
    history = []  # will store each day's step dictionary
    # Loop over days; enumerate gives (day_number, demand)
    for day, d in enumerate(demands, start=1):
        a = decide(s, target=target)  # DECIDE / policy → action a
        step = run_day(s, a, d)       # environment + feedback for this day
        step["day"] = day             # remember which day this was
        history.append(step)          # save the step
        s = step["s_next"]            # CRITICAL: tomorrow observes today's s'
    return history


# --- Demo the pipeline once for the room ---
demo = run_episode()  # run the default week
print("PIPELINE demo (observe → decide → next_state → reward → repeat)")
print("day  s  a  d  sales  r      s'")
# Print each day in a neat table
for step in demo:
    print(
        f"{step['day']:3} {step['s']:2} {step['a']:2} {step['d']:2} "
        f"{step['sales']:5} {step['r']:6.2f}  {step['s_next']}"
    )
# Teaching takeaway
print("Notice: each s' becomes the next row's s — that is the MDP memory.")


## Now check your hand math with the pipeline

> RUN WITH ME. Day 1, Day 2, and Day 3 must say **OK**.

| Day | $(s,a,d)$ | Hand $r$ | Story |
| --- | --- | --- | --- |
| 1 | $(0,0,2)$ | $-3.0$ | empty shelf stockout |
| 2 | $(1,2,2)$ | $3.8$ | order into stock |
| 3 | $(1,1,2)$ | $4.0$ | decide then clean sell |

This is your pencil work in code form.


In [ ]:
# ============================================================
# HAND-CHECK — RUN WITH ME (after DEFINE + PIPELINE)
# Goal: prove Days 1 and 2 from the Days 1–3 match the code.
# ============================================================

# Days 1–3 you already calculated
cases = [
    ("1", 0, 0, 2, -3.0),  # Day 1: empty shelf, no order, 2 customers
    ("2", 1, 2, 2, 3.8),   # Day 2: start 1, order 2, demand 2
    ("3", 1, 1, 2, 4.0),   # Day 3: decide(s=1)->a=1, demand 2
]

print("Hand → code check")
print(f"{'day':>3} {'s':>3} {'a':>3} {'d':>3} {'r_code':>7} {'r_hand':>7}  check")
for label, s, a, d, expected in cases:
    step = run_day(s, a, d)  # uses observe/next_state/reward under the hood
    ok = "OK" if abs(step["r"] - expected) < 1e-9 else "NOPE — recompute on paper"
    print(f"{label:>3} {s:>3} {a:>3} {d:>3} {step['r']:>7.2f} {expected:>7.2f}  {ok}")

print()
print("If all OK: your formulation is correct. Lab uses the same functions.")


# Part D — LAB TIME (10–15 minutes)

> **I walk around.** You use the **same DEFINE + PIPELINE** functions.  
> Change only `<<< CHANGE ME`. Raise a hand when the checklist is done.

| # | Challenge | Framework muscles |
| --- | --- | --- |
| 1 | One morning outcomes | `run_day` / `reward` / `next_state` |
| 2 | Best order chart | `list_actions` + `expected_reward` |
| 3 | Week stock path | `run_episode` (memory) |
| 4 | Change stockout penalty | `reward` knobs change `decide` quality |


## Challenge 1 — TRY THIS. One morning through the pipeline

Fix $s$ and $a$. For every $d\in\{0,1,2\}$, call `run_day(s,a,d)`.

Watch $r$ and $s'$ change with $d$. Also print $R(s,a)=\texttt{expected\_reward}(s,a)$.


In [ ]:
# ============================================================
# CHALLENGE 1 — TRY THIS
# Framework: fix s and a, then call run_day for every possible demand d.
# ============================================================

my_stock = 1   # <<< CHANGE ME — starting on-hand units (state s)
my_order = 1   # <<< CHANGE ME — morning order (action a)

# Show what we observed and what action we chose
print(f"observe(s)={observe(my_stock)}  action a={my_order}")
# Average score if we do not know demand yet
print(f"expected_reward(s,a)={expected_reward(my_stock, my_order):.3f}")
# Table header
print(f"{'d':>3} {'sales':>5} {'r':>7} {'s_next':>6}  story")

# For each possible demand and its probability from the real data
for d, p in sorted(DEMAND_P.items()):
    step = run_day(my_stock, my_order, d)                 # one full pipeline step
    started = min(CAPACITY, my_stock + my_order)          # units available that morning
    # Plain-English story for the row
    story = "STOCKOUT" if d > started else ("hold" if step["s_next"] else "cleared")
    # Print demand, sales, reward, next state, story, and probability
    print(f"{d:>3} {step['sales']:>5} {step['r']:>7.2f} {step['s_next']:>6}  {story}  (P={p:.3f})")


## Challenge 2 — TRY THIS. Which action looks best?

Observe one $s$. Score every $a\in\{0,1,2\}$ with $R(s,a)=\sum_d P(d)\,r(s,a,d)$.

Tallest bar = best morning order **for this state**.


In [ ]:
# ============================================================
# CHALLENGE 2 — TRY THIS
# Framework: observe one s, then score EVERY action in list_actions().
# ============================================================

start_stock = 1              # <<< CHANGE ME — try 0 or 2 too
s = observe(start_stock)     # OBSERVE → state s

# Expected reward for each legal action a
scores = [expected_reward(s, a) for a in list_actions()]
# Index of the best score → best action
best = list_actions()[int(np.argmax(scores))]

print(f"observe → s={s}")
# Print each decide candidate
for a, sc in zip(list_actions(), scores):
    mark = " ← best" if a == best else ""
    print(f"  decide candidate a={a}: expected r={sc:.3f}{mark}")

# --- bar chart: taller bar = better average morning for this s ---
fig, ax = plt.subplots(figsize=(6, 3.5))  # create a figure and axes
# Draw bars; crimson = best action, gold = others
colors = ["#A51417" if a == best else "#B6A36A" for a in list_actions()]
ax.bar([str(a) for a in list_actions()], scores, color=colors)
ax.set_title(f"Expected reward by action (s={s})")  # chart title
ax.set_xlabel("action a")                            # x-axis label
ax.set_ylabel("expected reward")                     # y-axis label
ax.axhline(0, color="black", linewidth=0.8)          # zero line for reference
plt.tight_layout()                                   # reduce label cutoff
plt.show()                                           # display the chart


## Challenge 3 — TRY THIS. Pipeline over a week (memory)

`run_episode` repeats observe → decide → run_day → $s\leftarrow s'$.

Partner check: point to `decide` and to $s'$ on the stock path.


In [ ]:
# ============================================================
# CHALLENGE 3 — TRY THIS
# Framework: run_episode = repeat observe → decide → run_day → s'
# ============================================================

target = 2       # <<< CHANGE ME — policy target for decide()
start_stock = 1  # <<< CHANGE ME — Monday morning stock

# Run the full multi-day PIPELINE
history = run_episode(start_stock=start_stock, target=target)

# Print the pipeline table
print("day  s  a  d  r     s'   ← s' becomes next s")
for step in history:
    print(f"{step['day']:3} {step['s']:2} {step['a']:2} {step['d']:2} {step['r']:6.2f} {step['s_next']:2}")

# Build a stock path: morning of day 1, then each s'
stocks = [history[0]["s"]] + [h["s_next"] for h in history]

# Plot stock over time (this line IS the MDP memory)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, len(stocks) + 1), stocks, marker="o", color="#A51417", linewidth=2)
ax.set_title(f"Stock path from run_episode (target={target})")
ax.set_xlabel("day index")
ax.set_ylabel("stock s")
ax.set_ylim(-0.2, CAPACITY + 0.2)  # keep shelf scale readable
ax.grid(True, alpha=0.3)           # light grid
plt.tight_layout()
plt.show()

print("Partner: point to decide() and to s' — those are framework slots, not magic.")


## Challenge 4 — TRY THIS. Change a DEFINE knob, rerun scores

Change `STOCKOUT_PENALTY` inside

$$
r=2.0\cdot\mathrm{sales}-0.2\cdot s'-\texttt{STOCKOUT\_PENALTY}\cdot\mathrm{stockout}
$$

Then recompute $R(0,a)$ for each $a$.


In [ ]:
# ============================================================
# CHALLENGE 4 — TRY THIS
# Framework: change a REWARD knob and see which ACTION looks best.
# ============================================================

_default = STOCKOUT_PENALTY          # remember class default so we can restore it
STOCKOUT_PENALTY = 1.5               # <<< CHANGE ME (try 0.0 soft, or 4.0 harsh)

s = observe(0)                       # OBSERVE empty shelf (stockouts matter a lot)
scores = [expected_reward(s, a) for a in list_actions()]  # score each action
best = list_actions()[int(np.argmax(scores))]             # best under THIS penalty

print(f"STOCKOUT_PENALTY={STOCKOUT_PENALTY} (default {_default})")
for a, sc in zip(list_actions(), scores):
    mark = " ← best under this reward" if a == best else ""
    print(f"  a={a}: {sc:.3f}{mark}")

# Chart the action comparison
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([str(a) for a in list_actions()], scores, color="#A51417")
ax.set_title(f"Actions when s=0, penalty={STOCKOUT_PENALTY}")
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

# Always restore the class default before later cells
STOCKOUT_PENALTY = _default
print(f"Restored STOCKOUT_PENALTY={STOCKOUT_PENALTY}")


## Lab checklist — raise a hand

1. [ ] I can name `observe`, `decide`, `next_state`, `reward`  
2. [ ] I ran Challenge 2 and can say which **action** wins  
3. [ ] I see $s'$ become next $s$ in Challenge 3  
4. [ ] Challenge 4 changed the **reward** story  

Raise a hand when done (or stuck).


In [ ]:
# ============================================================
# LAB SELF-CHECK — optional
# Goal: confirm DEFINE + PIPELINE pieces still work after exploring.
# ============================================================

# Each item is (label, True/False test)
checks = [
    ("load_dataset keys", set(DEMAND_P) == {0, 1, 2}),                 # demand model has 0,1,2
    ("observe", observe(3) == 3),                                      # observe returns stock
    ("list_actions", list_actions() == [0, 1, 2]),                      # legal actions
    ("decide slot", decide(1) in list_actions()),                       # decide returns a legal a
    ("run_day keys", set(run_day(1, 1, 1)) >= {"s", "a", "d", "r", "s_next"}),  # pipeline fields
    ("Day 1 hurt", abs(reward(0, 0, 2) + 3.0) < 1e-9),                 # empty-shelf stockout = -3
]

# Print OK/FAIL for each test
for name, ok in checks:
    print(("OK  " if ok else "FAIL"), name)

# Reminder for the room
print("Raise a hand if all OK — or if any FAIL.")


# Part E — Come back together

We defined the framework, ran the pipeline, then explored.

**Recap aloud (30 seconds)**

1. Notation: $s,a,d,s',r,C,P(d),\gamma$
2. Physics: $\mathrm{started},\mathrm{sales},s',\mathrm{stockout}$
3. Day 1 must hurt: $r=-3.0$
4. Pipeline: observe → decide → next_state → reward → $s\leftarrow s'$


## Together — empty shelf must hurt

Guess $r$ for `run_day(0, 0, 2)` then run.

**Day 1 reminder**

$$
\mathrm{started}=0,\;\mathrm{sales}=0,\;s'=0,\;\mathrm{stockout}=2,\;
r=2.0\cdot 0-0.2\cdot 0-1.5\cdot 2=-3.0
$$


In [ ]:
# ============================================================
# TOGETHER — FILL
# Goal: guess the pipeline reward before trusting the code.
# Setup: s=0, a=0, d=2 → should hurt (−3.0).
# ============================================================

expected_reward_guess = None  # <<< YOU TYPE HERE (should be -3.0)

# Run the same case through the PIPELINE
step = run_day(0, 0, 2)

# Compare your guess to the pipeline output
print("You expected", expected_reward_guess, "| pipeline r=", round(step["r"], 2), "| s'=", step["s_next"])


## Bridge + framework growth

Session 6 uses the **same** define → pipeline idea for an LLM design and an MDP design, then a go/no-go memo.

Tonight we **added** to `Agent-Framework.md`: `observe`, `list_actions`, `decide`, `next_state`, `reward`, `run_day`, `run_episode`.  
Later sessions add Q-updates / DQN **into** `decide` — they do not throw this away.


## Framework sentence — OPEN · STOP

Family for **your** project + one keepable sentence (use the words observe / decide / reward / next state if you can).


In [ ]:
# ============================================================
# FRAMEWORK — FILL
# Goal: leave with a family label + one sentence you will reuse.
# ============================================================

framework_family = ""  # <<< YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # <<< YOU TYPE HERE: one sentence (use observe/decide/reward/next state if you can)

print("Family:", framework_family or "(type it)")
print("I will keep:", framework_line or "(type it)")
